In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

In [2]:

import pandas as pd
import numpy as  np
import datetime as dt
import pandas as pd
import numpy as np
import datetime as dt

from Python_arq import engines as engs
from sqlalchemy import text
from calendar import day_name
from sqlalchemy import text
from pathlib import Path


eng = engs.get_engine()
text_qry = text(engs.load_query('qry_olos.sql'))
df = pd.read_sql(text_qry, eng)


Carregando query: qry_olos.sql


In [3]:
dia_semana = {
    'Monday': 'Segunda',
    'Tuesday': 'Terça',
    'Wednesday': 'Quarta',
    'Thursday': 'Quinta',
    'Friday': 'Sexta',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df['day_week'] = df['data'].apply(lambda x: day_name[x.weekday()]).map(dia_semana)
df['data'] = pd.to_datetime(df['data'])
df['semana_mes'] = (
    df['data'].dt.day.sub(1) // 7 + 1 
)
df['day_week'] = df['day_week'] + '_W' + df['semana_mes'].astype(str)

In [4]:
## performance bases | Hoje ## 
df_today = df[df['data'].dt.date == dt.datetime.today().date()]
df_today = df_today.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_today['performance'] = (df_today['total_atendidas'] / df_today['Total_tentativas'] * 100).round(2)
df_today = df_today.sort_values('performance', ascending=False)
df_today

,area,base_type,Total_tentativas,total_atendidas,performance
5,Psicologia,Material Rico,2065,45,2.18
0,Enfermagem,Material Rico,1799,36,2.00
6,Psicologia,ativa,771,15,1.95
1,Fisioterapia,Material Rico,852,16,1.88
4,Medicina,ativa,2765,50,1.81
3,Medicina,Material Rico,3060,51,1.67
7,Psicologia,inativa,1993,23,1.15
2,Fisioterapia,ativa,4510,47,1.04


In [12]:
## top10 bases ## 
df_top10 = df.groupby(['area','base_type']).agg(
    total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_top10 = df_top10[df_top10['total_tentativas']>=1000]
df_top10['performance'] = (df_top10['total_atendidas'] / df_top10['total_tentativas'] * 100).round(2)
df_top10 = df_top10.sort_values('performance', ascending=False).head(10)
df_top10

,area,base_type,total_tentativas,total_atendidas,performance
28,Multi,evento,10897,467,4.29
26,Multi,Material Rico,3556,149,4.19
36,Pediatria,Base Leads,3153,131,4.15
40,Pediatria,evento,4805,189,3.93
31,Nutrição,Base Leads,3855,150,3.89
5,Enfermagem,evento,30430,1110,3.65
35,Outras Áreas,evento,4761,162,3.40
7,Enfermagem,legado,13305,448,3.37
33,Nutrição,evento,24514,790,3.22
47,Psicologia,evento,19852,627,3.16


In [6]:
## performance bases | D-1 ## 
df_ontem = df[df['data'] == '2026-04-14']
df_ontem = df_ontem.groupby(['area','base_type']).agg(
    Total_tentativas = ('tentativas','sum'),
    total_atendidas = ('atendidas','sum')
).reset_index()
df_ontem['performance'] = (df_ontem['total_atendidas'] / df_ontem['Total_tentativas'] * 100).round(2)
df_ontem = df_ontem.sort_values('performance', ascending=False)
df_ontem

,area,base_type,Total_tentativas,total_atendidas,performance
0,Enfermagem,Material Rico,1406,39,2.77
3,Fisioterapia,evento,1717,44,2.56
4,Medicina,Base Leads,2235,48,2.15
9,Psicologia,inativa,722,15,2.08
1,Fisioterapia,Material Rico,4665,93,1.99
8,Psicologia,ativa,1011,16,1.58
6,Medicina,ativa,4291,63,1.47
2,Fisioterapia,ativa,3451,49,1.42
7,Psicologia,Material Rico,2211,31,1.40
5,Medicina,Material Rico,2957,41,1.39


In [7]:
bases_atuadas = df.groupby(['area','base_type','tablename','data']).agg(
    tentativas = ('tentativas','sum'),
    atendidas = ('atendidas','sum')
).reset_index()
bases_atuadas['split_tablename'] = bases_atuadas['tablename'].str.split('_')
bases_atuadas = bases_atuadas.to_excel(r'C:\Users\wconceicao\OneDrive - Grupo A Educação SA\Área de Trabalho\bases_atuadas.xlsx')

In [8]:
df

,area,base_type,campaign_id,tablename,data,hour,tentativas,atendidas,day_week,semana_mes
0,Fisioterapia,cancelados,1605,_1605_FISIOTERAPIA_CANCELADO_0_27022026_V513_pt5,2026-02-27,18.0,31,1,Sexta_W4,4
1,Psicologia,ativa,1299,_1299_PSICOLOGIA_ATIVA_OFERTARPRONEUROPSI_0704...,2026-04-07,11.0,222,11,Terça_W1,1
2,Fisioterapia,None,1605,_1605_FISIOTERAPIA_PAGINAPRODUTO_PROFISIOTO_0_...,2025-11-24,16.0,95,2,Segunda_W4,4
3,Enfermagem,cancelados,1605,_1605_ENFERMAGEM_CANCELADO_16032026_V500_PT1,2026-03-16,10.0,739,5,Segunda_W3,3
4,Enfermagem,inativa,1605,_1605_ENFERMAGEM_INATIVO_0_08012026_V10563,2026-01-09,11.0,27,1,Sexta_W2,2
...,...,...,...,...,...,...,...,...,...,...
9713,Fisioterapia,cancelados,1605,_1605_FISIOTERAPIA_CANCELADO_0_27022026_V513_pt3,2026-03-02,11.0,27,1,Segunda_W1,1
9714,Fisioterapia,Base Leads,1605,_1605_FISIOTERAPIA_POS_BASELEADS_26112025PT1_V...,2025-11-26,12.0,168,11,Quarta_W4,4
9715,Psicologia,Base Leads,1605,_1605_PSICOLOGIA_POSARTMED_BASELEADS_MATRICULA...,2026-04-02,11.0,111,0,Quinta_W1,1
9716,Medicina,inativa,1605,_1605_MEDICINA_INATIVO_0_03022026_V6187,2026-02-05,9.0,353,6,Quinta_W1,1
